In [1]:
# Ultra-accuracy YOLOv5 + SORT + temporal smoothing + EMA bbox + robust alarm
# Works with your existing environment (torch, cv2, numpy, sort.py, winsound)

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger().setLevel(logging.ERROR)

import cv2
import torch
import numpy as np
from sort import Sort
import winsound
import threading
from collections import deque, defaultdict

# -----------------------------
# Config (tweak these if needed)
# -----------------------------
WEIGHTS_PATH = r'yolov5/runs/train/exp12/weights/best.pt'  # change if different
CONF_THRESHOLD = 0.55      # per-frame YOLO confidence filter (higher -> stricter)
AVG_CONF_THRESHOLD = 0.30  # average drowsy confidence threshold to consider (soft)
SUSTAIN_FRAMES = 18        # consecutive averaged frames needed to trigger alarm (~0.6s at 30fps)
WINDOW = 30                # sliding window size for averaging
EMA_ALPHA = 0.35           # bbox smoothing factor (0-1, higher => more responsive)
MIN_BOX_AREA = 900         # minimum bbox area (width*height) to accept detection
DROWSY_CLASS_ID = 1        # class index for 'drowsy' (0 = awake, 1 = drowsy)
AWAKE_CLASS_ID = 0

# -----------------------------
# Load YOLOv5 (silent)
# -----------------------------
model = torch.hub.load('ultralytics/yolov5', 'custom', path=WEIGHTS_PATH, verbose=False)
model.conf = CONF_THRESHOLD
model.iou = 0.45
model.max_det = 6

# -----------------------------
# Initialize tracker & state
# -----------------------------
tracker = Sort()              # your simple sort.py
per_track_window = defaultdict(lambda: deque(maxlen=WINDOW))
per_track_sustain = defaultdict(int)
per_track_bbox_ema = {}       # track_id -> [x1,y1,x2,y2] EMA smoothed bbox
alarm_flags = defaultdict(bool)
alarm_lock = threading.Lock()

# Alarm function (winsound beep)
def play_alarm(track_id):
    with alarm_lock:
        # simple single beep; replace with loop if you want continuous sound
        winsound.Beep(2200, 900)
        alarm_flags[track_id] = False

# Utility: compute center
def center_of_bbox(b):
    x1, y1, x2, y2 = b
    return int((x1 + x2) // 2), int((y1 + y2) // 2)

# Utility: safe draw text with background
def draw_text(img, text, org, color=(0,255,0), scale=0.6, thickness=2):
    x,y = org
    (w,h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    cv2.rectangle(img, (x-2, y-2), (x+w+2, y+h+2), (0,0,0), -1)
    cv2.putText(img, text, (x, y+h), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)

# -----------------------------
# Start webcam loop
# -----------------------------
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam (0).")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    results = model(frame)                       # inference
    df = results.pandas().xyxy[0]                # dataframe: xmin,ymin,xmax,ymax,confidence,class,...

    # Prepare detections for SORT: [x1,y1,x2,y2,conf]
    dets = []
    det_meta = []  # store metadata per detection for later label/class/conf mapping
    for _, row in df.iterrows():
        x1, y1, x2, y2 = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
        conf = float(row['confidence'])
        cls = int(row['class'])
        area = max(0, (x2 - x1)) * max(0, (y2 - y1))
        # area & bounds check
        if area < MIN_BOX_AREA or x2<=x1 or y2<=y1:
            continue
        # keep detection
        dets.append([x1, y1, x2, y2, conf])
        det_meta.append({'bbox':(x1,y1,x2,y2), 'conf':conf, 'class':cls})

    dets_np = np.array(dets) if len(dets) else np.empty((0,5))

    # Update SORT tracker
    tracked = tracker.update(dets_np)   # returns array of [x1,y1,x2,y2,track_id] or empty

    # Build mapping: track_id -> matched detection (best by center distance)
    # Precompute detection centers
    det_centers = [center_of_bbox(d['bbox']) for d in det_meta]

    # For robust matching, we'll compute center distances between tracked bbox center and each detection center
    track_to_det = {}  # track_id -> index in det_meta

    for tr in tracked:
        tx1, ty1, tx2, ty2, track_id = tr
        tx1, ty1, tx2, ty2, track_id = int(tx1), int(ty1), int(tx2), int(ty2), int(track_id)
        tcenter = ((tx1 + tx2) // 2, (ty1 + ty2) // 2)

        # find nearest detection center within tolerance
        best_idx, best_dist = None, 1e9
        for i, dc in enumerate(det_centers):
            dist = abs(dc[0] - tcenter[0]) + abs(dc[1] - tcenter[1])
            if dist < best_dist:
                best_dist = dist
                best_idx = i
        # require that the distance is reasonably small (tolerance relative to box size)
        box_w = max(1, tx2 - tx1)
        tol = max(40, box_w // 2)
        if best_dist <= tol:
            track_to_det[track_id] = best_idx
        else:
            # no reliable matching - skip label update but still draw smoothed box
            track_to_det[track_id] = None

    # For each tracked object, update per-track sliding window & EMA
    for tr in tracked:
        tx1, ty1, tx2, ty2, track_id = tr
        tx1, ty1, tx2, ty2, track_id = int(tx1), int(ty1), int(tx2), int(ty2), int(track_id)

        # EMA bbox smoothing
        if track_id in per_track_bbox_ema:
            prev = per_track_bbox_ema[track_id]
            smoothed = [
                int(EMA_ALPHA * tx1 + (1-EMA_ALPHA) * prev[0]),
                int(EMA_ALPHA * ty1 + (1-EMA_ALPHA) * prev[1]),
                int(EMA_ALPHA * tx2 + (1-EMA_ALPHA) * prev[2]),
                int(EMA_ALPHA * ty2 + (1-EMA_ALPHA) * prev[3]),
            ]
            per_track_bbox_ema[track_id] = smoothed
        else:
            per_track_bbox_ema[track_id] = [tx1, ty1, tx2, ty2]

        # Find matched detection (if any) and update sliding window
        det_idx = track_to_det.get(track_id, None)
        if det_idx is not None and det_idx < len(det_meta):
            meta = det_meta[det_idx]
            cls = meta['class']
            conf = meta['conf']
            # we will store drowsy_conf value (if detection class is drowsy -> its conf, else 0)
            drowsy_conf = conf if cls == DROWSY_CLASS_ID else 0.0
            # also store awake_conf for visualization (not used for alarm)
            awake_conf = conf if cls == AWAKE_CLASS_ID else 0.0
            per_track_window[track_id].append({'drowsy': drowsy_conf, 'raw_cls': cls, 'conf': conf})
        else:
            # no detection matched this tracker this frame -> append zeros (no detection)
            per_track_window[track_id].append({'drowsy': 0.0, 'raw_cls': None, 'conf': 0.0})

        # compute rolling statistics
        window = per_track_window[track_id]
        if len(window):
            avg_drowsy = float(np.mean([x['drowsy'] for x in window]))
            # count consecutive frames where per-frame raw_cls == drowsy at end of deque
            cons = 0
            for rec in reversed(window):
                if rec['raw_cls'] == DROWSY_CLASS_ID:
                    cons += 1
                else:
                    break
        else:
            avg_drowsy = 0.0
            cons = 0

        # decide status
        is_drowsy_soft = (avg_drowsy >= AVG_CONF_THRESHOLD)
        is_drowsy_sustained = (cons >= SUSTAIN_FRAMES)

        # Alarm logic: require both soft average and sustained frames (very robust)
        if is_drowsy_soft and is_drowsy_sustained and not alarm_flags[track_id]:
            alarm_flags[track_id] = True
            # start alarm thread
            threading.Thread(target=play_alarm, args=(track_id,), daemon=True).start()

        # Stop alarm if condition no longer holds (optional immediate stop)
        if not is_drowsy_soft or not is_drowsy_sustained:
            # allow current alarm to finish but clear flag to allow future alarms
            alarm_flags[track_id] = False

        # Visualization values
        bbox_vis = per_track_bbox_ema[track_id]
        x1v, y1v, x2v, y2v = map(int, bbox_vis)
        label_text = f"ID:{track_id}"
        if is_drowsy_soft:
            label_text += f" Drowsy(avg:{avg_drowsy:.2f} cons:{cons})"
            color = (0,0,255)
        else:
            label_text += f" Awake(avg:{avg_drowsy:.2f})"
            color = (0,255,0)

        # Draw smoothed box and label
        cv2.rectangle(frame, (x1v, y1v), (x2v, y2v), color, 2)
        draw_text(frame, label_text, (x1v, y1v-20), color=color, scale=0.55, thickness=1)

    # show FPS
    cv2.putText(frame, f"Press ESC to quit", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)
    cv2.imshow("Ultra Accuracy Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


YOLOv5  2025-12-13 Python-3.10.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
